# Statcast 2023-2024 데이터 탐색

SmartPitch MDP 전이확률 모델 비교를 위한 데이터 탐색 노트북.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 11

In [ ]:
df_2023 = pd.read_parquet("../data/raw/2023/statcast_2023.parquet")
df_2024 = pd.read_parquet("../data/raw/2024/statcast_2024.parquet")
df = pd.concat([df_2023, df_2024], ignore_index=True)
df["game_date"] = pd.to_datetime(df["game_date"])
df["season"] = df["game_date"].dt.year
print(f"2023: {len(df_2023):,} rows")
print(f"2024: {len(df_2024):,} rows")
print(f"Combined: {df.shape[0]:,} rows × {df.shape[1]} columns")

In [ ]:
print(f"Shape: {df.shape}")
print(f"\nDtype distribution:")
print(df.dtypes.value_counts())
print(f"\nMemory usage: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")

In [ ]:
null_pct = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)
null_cols = null_pct[null_pct > 0]

# 100% 결측 = deprecated 컬럼 식별
deprecated = null_pct[null_pct >= 99.9].index.tolist()
print(f"Deprecated columns (≥99.9% null): {deprecated}\n")

fig, ax = plt.subplots(figsize=(10, 10))
null_cols.head(40).plot.barh(ax=ax, color="coral")
ax.set_xlabel("Missing ratio (%)")
ax.set_title(f"Column null ratio (top 40 of {len(null_cols)} with nulls)")
ax.invert_yaxis()
plt.tight_layout()

In [ ]:
cat_cols = ["pitch_type", "events", "description", "type", "zone", "stand", "p_throws"]

for col in cat_cols:
    if col not in df.columns:
        continue
    vc = df[col].value_counts(dropna=False)
    n_unique = df[col].nunique()
    null_n = df[col].isnull().sum()
    print(f"\n{'='*50}")
    print(f"{col} — {n_unique} unique, {null_n:,} nulls ({null_n/len(df)*100:.1f}%)")
    print(vc.to_string())

In [ ]:
batter_counts = df.groupby("batter").size().rename("n_pitches")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 전체 분포
batter_counts.hist(bins=80, ax=axes[0], color="steelblue", edgecolor="white")
axes[0].axvline(400, color="red", ls="--", lw=1.5, label="400 pitches")
axes[0].axvline(200, color="orange", ls="--", lw=1.5, label="200 pitches")
axes[0].set_xlabel("Pitches per batter")
axes[0].set_ylabel("Count")
axes[0].set_title("All batters")
axes[0].legend()

# 400+ 확대
batter_400 = batter_counts[batter_counts >= 400]
batter_400.hist(bins=40, ax=axes[1], color="seagreen", edgecolor="white")
axes[1].set_xlabel("Pitches per batter")
axes[1].set_title(f"Batters with 400+ pitches (n={len(batter_400)})")

plt.tight_layout()

print(f"Total batters: {len(batter_counts):,}")
print(f"Mean: {batter_counts.mean():.1f}, Median: {batter_counts.median():.1f}")
print(f"400+ pitches: {(batter_counts >= 400).sum()}")
print(f"200-400 pitches: {((batter_counts >= 200) & (batter_counts < 400)).sum()}")
print(f"<200 pitches: {(batter_counts < 200).sum()}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# pitch_type 분포
ct = pd.crosstab(df["pitch_type"], df["season"])
ct_top = ct.loc[ct.sum(axis=1).nlargest(10).index]
ct_top.plot.bar(ax=axes[0])
axes[0].set_title("Pitch type by season (top 10)")
axes[0].set_ylabel("Count")

# type (B/S/X) 비율
ct_type = pd.crosstab(df["type"], df["season"], normalize="columns") * 100
ct_type.plot.bar(ax=axes[1])
axes[1].set_title("Outcome type ratio by season (%)")
axes[1].set_ylabel("Percentage")

# 월별 pitch 수
df["month"] = df["game_date"].dt.month
monthly = df.groupby(["season", "month"]).size().unstack("season")
monthly.plot(ax=axes[2], marker="o")
axes[2].set_title("Monthly pitch counts")
axes[2].set_xlabel("Month")
axes[2].set_ylabel("Pitches")

plt.tight_layout()

In [ ]:
num_cols = [
    "release_speed", "release_spin_rate", "pfx_x", "pfx_z",
    "plate_x", "plate_z", "launch_speed", "launch_angle",
]

fig, axes = plt.subplots(2, 4, figsize=(20, 8))
for ax, col in zip(axes.flat, num_cols):
    data = df[col].dropna()
    ax.hist(data, bins=60, alpha=0.7, color="steelblue", edgecolor="white")
    ax.axvline(data.mean(), color="red", ls="--", lw=1, label=f"μ={data.mean():.1f}")
    ax.set_title(f"{col}\n(null {df[col].isnull().mean()*100:.1f}%)")
    ax.legend(fontsize=8)

plt.suptitle("Core feature distributions", fontsize=14, y=1.01)
plt.tight_layout()